In [5]:
import json
from datasets import load_dataset, Audio
from huggingface_hub import snapshot_download

jsonl = snapshot_download('ddwang2000/MMSU', repo_type="dataset", allow_patterns="*.jsonl")
jsonl = jsonl + "/question/mmsu.jsonl"

with open(jsonl, "r") as f:
    jsonl_datas = []
    for line in f.readlines():
        jsonl_datas.append(json.loads(line))

ds = load_dataset('ddwang2000/MMSU')["train"]

ds = ds.cast_column("audio", Audio(decode=False))

ds, jsonl_datas

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/5001 [00:00<?, ?it/s]

(Dataset({
     features: ['audio'],
     num_rows: 5000
 }),
 [{'id': 'volume_comparison_6b58eff0-f0ff-4558-89e9-52ca0ed489bf',
   'task_name': 'volume_comparison',
   'audio_path': '/audio/volume_comparison_6b58eff0-f0ff-4558-89e9-52ca0ed489bf.wav',
   'question': 'Which volume pattern best matches the audio?',
   'choice_a': 'high-low-medium',
   'choice_b': 'medium-high-low',
   'choice_c': 'low-high-medium',
   'choice_d': 'low-medium-high',
   'answer_gt': 'high-low-medium',
   'category': 'Perception',
   'sub-category': 'Paralinguistics',
   'sub-sub-category': 'Speaker Traits',
   'linguistics_sub_discipline': 'Paralinguistics'},
  {'id': 'volume_comparison_42e13b39-6c6c-43bd-b94d-598b754fa96b',
   'task_name': 'volume_comparison',
   'audio_path': '/audio/volume_comparison_42e13b39-6c6c-43bd-b94d-598b754fa96b.wav',
   'question': 'Which volume pattern best matches the audio?',
   'choice_a': 'low-high-medium',
   'choice_b': 'medium-low-high',
   'choice_c': 'medium-high-low'

In [6]:

jsonl_datas_dict = {}
for data in jsonl_datas:
    jsonl_datas_dict[data["id"]] = data

def map_func(item):
    path = item["audio"]["path"]
    r = jsonl_datas_dict[path.split("/")[-1].split(".")[0]]

    answer_index = 0
    if r["answer_gt"] == r["choice_a"]:
        answer_index = 0
    elif r["answer_gt"] == r["choice_b"]:
        answer_index = 1
    elif r["answer_gt"] == r["choice_c"]:
        answer_index = 2
    elif r["answer_gt"] == r["choice_d"]:
        answer_index = 3
    
    r["answer_index"] = answer_index
    r["options"] = [str(r["choice_a"]), str(r["choice_b"]), str(r["choice_c"]), str(r["choice_d"])]

    del r["choice_a"]
    del r["choice_b"]
    del r["choice_c"]
    del r["choice_d"]
    del r["answer_gt"]
    del r["audio_path"]
    return r

ds = ds.map(map_func, writer_batch_size=200)
ds

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'id', 'task_name', 'question', 'category', 'sub-category', 'sub-sub-category', 'linguistics_sub_discipline', 'answer_index', 'options'],
    num_rows: 5000
})

In [7]:
ds = ds.rename_column("id", "key")
ds[0]

{'audio': {'bytes': None,
  'path': '/home/aiscuser/.cache/huggingface/hub/datasets--ddwang2000--MMSU/snapshots/abe42e386504018657c70fa95302f95cc0d5c186/audio/accent_identification_007d014e-d739-4984-b03c-5ae4594adc5e.wav'},
 'key': 'accent_identification_007d014e-d739-4984-b03c-5ae4594adc5e',
 'task_name': 'accent_identification',
 'question': "What accent does the speaker's voice most likely correspond to?",
 'category': 'Perception',
 'sub-category': 'Linguistics',
 'sub-sub-category': 'Phonology',
 'linguistics_sub_discipline': 'Prosody',
 'answer_index': 0,
 'options': ['United States', 'Singapore', 'Australia', 'Nigeria']}

In [8]:
ds = ds.map(lambda item: {
    'category': item["category"]+"-"+item["sub-category"]+"-"+item["sub-sub-category"]+"-"+item["linguistics_sub_discipline"],
    'prompt_for_tts': f"{item['question']}\nA. {item['options'][0]}\nB. {item['options'][1]}\nC. {item['options'][2]}\nD. {item['options'][3]}"
}, remove_columns=["sub-category", "sub-sub-category", "task_name"])
ds[0]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

{'audio': {'bytes': None,
  'path': '/home/aiscuser/.cache/huggingface/hub/datasets--ddwang2000--MMSU/snapshots/abe42e386504018657c70fa95302f95cc0d5c186/audio/accent_identification_007d014e-d739-4984-b03c-5ae4594adc5e.wav'},
 'key': 'accent_identification_007d014e-d739-4984-b03c-5ae4594adc5e',
 'question': "What accent does the speaker's voice most likely correspond to?",
 'category': 'Perception-Linguistics-Phonology-Prosody',
 'linguistics_sub_discipline': 'Prosody',
 'answer_index': 0,
 'options': ['United States', 'Singapore', 'Australia', 'Nigeria'],
 'prompt_for_tts': "What accent does the speaker's voice most likely correspond to?\nA. United States\nB. Singapore\nC. Australia\nD. Nigeria"}

In [9]:
ds = ds.cast_column("audio", Audio())
ds[0]

{'audio': {'path': '/home/aiscuser/.cache/huggingface/hub/datasets--ddwang2000--MMSU/snapshots/abe42e386504018657c70fa95302f95cc0d5c186/audio/accent_identification_007d014e-d739-4984-b03c-5ae4594adc5e.wav',
  'array': array([0.00000000e+00, 9.33718167e-17, 4.98140407e-17, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
  'sampling_rate': 48000},
 'key': 'accent_identification_007d014e-d739-4984-b03c-5ae4594adc5e',
 'question': "What accent does the speaker's voice most likely correspond to?",
 'category': 'Perception-Linguistics-Phonology-Prosody',
 'linguistics_sub_discipline': 'Prosody',
 'answer_index': 0,
 'options': ['United States', 'Singapore', 'Australia', 'Nigeria'],
 'prompt_for_tts': "What accent does the speaker's voice most likely correspond to?\nA. United States\nB. Singapore\nC. Australia\nD. Nigeria"}

In [10]:
ds.save_to_disk(os.environ.get("CUHK_MMSU_DS_PATH", "/path/to/your/dataset/MMSU/full_5k_hf_format.v0"))

Saving the dataset (0/3 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]